# Collection — Adoption of context files and content download

Reproduces the subsection *Presence of context files* (RQ1) and the section *Data Collection* of the paper:

| Result in the paper | Where |
|---|---|
| 228,890 SEART repositories; 83 archived in the top 10,000; 62.4% overlap with the sample of Mohsenimofidi et al. | section 0 |
| 1,256 of 10,000 repositories (12.6%) with at least one file | section 1 |
| 769 / 382 / 95 / 10 repositories with 1 / 2 / 3 / 4 files; 487 (38.8%) with more than one | section 1 |
| Table of formats (1,858 files) | section 2 |
| 1,828 files downloaded, 30 failures (1.6%) | section 3 |
| 58.4% of the downloaded files coexist with another file in the same repository | section 3 |
| Motivating example: size of the two files of `dotcms/core` | section 4 |

Inputs: `data/collection/seart/seart_24-08-2025_names_archived.csv.gz` (name and `isArchived` of each
repository in the SEART export), `data/collection/ranking/` (the study's top 10,000 and the list of
Mohsenimofidi et al.), `data/collection/detection/detection_top10000_2026-05-07.csv` (output of
`scripts/detect_ai_context_files/run_detect_context.py`), and
`data/collection/context_files/download_manifest.csv` (output of `scripts/download_ai_context_files.py`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figures import save, svg_formats_table

SEART = ROOT / "data/collection/seart/seart_24-08-2025_names_archived.csv.gz"
TOP = ROOT / "data/collection/ranking/top10000_24-08-2025.csv"
REFERENCE = ROOT / "data/collection/ranking/mohsenimofidi_top10000.txt"
DETECTION = ROOT / "data/collection/detection/detection_top10000_2026-05-07.csv"
MANIFEST = ROOT / "data/collection/context_files/download_manifest.csv"
OUT = ROOT / "results/collection"
OUT.mkdir(parents=True, exist_ok=True)

FORMATS = ["AGENTS.md", "CLAUDE.md", "COPILOT", "GEMINI.md", "CURSOR"]

## 0. Repository selection

The ranking drops 126 repositories that lack an essential field (`ranking.py`), so 228,764 of the
228,890 repositories are ranked.

In [2]:
seart = pd.read_csv(SEART)
top = pd.read_csv(TOP, usecols=["name"])
reference = {l.strip().lower() for l in REFERENCE.read_text().splitlines() if l.strip()}
archived = int(seart.set_index("name")["isArchived"].reindex(top["name"]).astype(bool).sum())
overlap = len(reference & set(top["name"].str.lower()))
print(f"Repositories in the SEART export: {len(seart):,}")
print(f"Archived in the top 10,000: {archived}")
print(f"Overlap with the top 10,000 of Mohsenimofidi et al.: {overlap:,} ({100 * overlap / len(top):.1f}%)")

Repositories in the SEART export: 228,890
Archived in the top 10,000: 83
Overlap with the top 10,000 of Mohsenimofidi et al.: 6,239 (62.4%)


## 1. Adoption

In [3]:
det = pd.read_csv(DETECTION)
det["n_files"] = det[FORMATS].sum(axis=1)
adopters = det[det["n_files"] > 0]

n_repos, n_adopters = len(det), len(adopters)
dist = adopters["n_files"].value_counts().sort_index()
n_multi = int((adopters["n_files"] >= 2).sum())

print(f"Repositories analyzed: {n_repos:,}")
print(f"With at least one file: {n_adopters:,} ({100 * n_adopters / n_repos:.1f}%)")
print(pd.DataFrame({"repositories": dist, "pct_of_adopters": (100 * dist / n_adopters).round(1)}))
print(f"With more than one file: {n_multi} ({100 * n_multi / n_adopters:.1f}% of adopters)")

Repositories analyzed: 10,000
With at least one file: 1,256 (12.6%)
         repositories  pct_of_adopters
n_files                               
1                 769             61.2
2                 382             30.4
3                  95              7.6
4                  10              0.8
With more than one file: 487 (38.8% of adopters)


## 2. Table of formats

In [4]:
TOOL = {"AGENTS.md": "Open standard", "CLAUDE.md": "Claude Code", "COPILOT": "GitHub Copilot",
        "GEMINI.md": "Gemini CLI", "CURSOR": "Cursor"}
NAME = {"COPILOT": "copilot-instructions.md", "CURSOR": ".cursorrules"}

counts = det[FORMATS].sum()
n_files = int(counts.sum())
table = pd.DataFrame({"format": [NAME.get(f, f) for f in FORMATS],
                      "tool": [TOOL[f] for f in FORMATS],
                      "files": counts.values,
                      "pct": 100 * counts.values / n_files})
table.loc[len(table)] = {"format": "Total", "tool": "", "files": n_files, "pct": 100.0}
table.to_csv(OUT / "formats_table.csv", index=False)
save(svg_formats_table(table), OUT / "formats-table.svg")
table.round(2)

saved /home/keven/Documents/tcc/repo-pesquisa-repro/results/collection/formats-table.svg


,format,tool,files,pct
0,AGENTS.md,Open standard,709,38.16
1,CLAUDE.md,Claude Code,657,35.36
2,copilot-instructions.md,GitHub Copilot,418,22.50
3,GEMINI.md,Gemini CLI,42,2.26
4,.cursorrules,Cursor,32,1.72
5,Total,,1858,100.00


## 3. Content download

We check whether the 30 failures are concentrated in some format. As there are few failures per
format, the p-value of the chi-square test is obtained by permutation (20,000 draws, fixed seed).

In [5]:
man = pd.read_csv(MANIFEST, keep_default_na=False)
ok = man[man["status"] == "ok"]
failed = man[man["status"] != "ok"]
print(f"Files detected: {len(man):,}; downloaded: {len(ok):,}; failures: {len(failed)} "
      f"({100 * len(failed) / len(man):.1f}%)")

by_format = pd.crosstab(man["file_label"], man["status"]).reindex(FORMATS)
by_format["pct_failed"] = (100 * by_format["failed"] / by_format.sum(axis=1)).round(1)
print(by_format)


def chi2_failures(status):
    return chi2_contingency(pd.crosstab(man["file_label"], status).reindex(FORMATS).values)[0]


is_failure = (man["status"] != "ok").values
chi2_obs = chi2_failures(is_failure)
rng = np.random.default_rng(42)
n_perm = 20_000
extreme = sum(chi2_failures(rng.permutation(is_failure)) >= chi2_obs for _ in range(n_perm))
p_perm = (extreme + 1) / (n_perm + 1)
print(f"Failures × format: χ² = {chi2_obs:.2f}, p (permutation) = {p_perm:.4f}")

files_per_repo = ok.groupby("repo").size()
coexist = ok["repo"].map(files_per_repo) >= 2
print(f"Downloaded files that coexist with another file in the same repository: {100 * coexist.mean():.1f}%")

Files detected: 1,858; downloaded: 1,828; failures: 30 (1.6%)
status      failed   ok  pct_failed
file_label                         
AGENTS.md        6  703         0.8
CLAUDE.md        8  649         1.2
COPILOT          9  409         2.2
GEMINI.md        5   37        11.9
CURSOR           2   30         6.2


Failures × format: χ² = 36.37, p (permutation) = 0.0002
Downloaded files that coexist with another file in the same repository: 58.4%


## 4. Motivating example

In [6]:
for name in ["CLAUDE.md", "copilot-instructions.md"]:
    text = (ROOT / "data/collection/context_files/dotcms__core" / name).read_text(encoding="utf-8")
    print(f"dotcms/core/{name}: {len(text.splitlines())} lines")

dotcms/core/CLAUDE.md: 144 lines
dotcms/core/copilot-instructions.md: 884 lines
